In [1]:
!pip install pymupdf pandas tqdm openai python-dotenv rapidfuzz


In [2]:
import json
import os
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()


def find_workspace_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datasets" / "training" / "Books").exists():
            return candidate
    return start.resolve()


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
BOOKS_DIR = WORKSPACE_ROOT / "datasets" / "training" / "Books"
OUTPUT_DIR = WORKSPACE_ROOT / "datasets" / "training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# QWEN provider for question generation
QWEN_BASE_URL = os.getenv("QWEN_LLM_API_URL", "").strip()
QWEN_MODEL = os.getenv("QWEN_LLM_MODEL_NAME", "Qwen/Qwen3.5-27B").strip()
QWEN_API_KEY = (
    os.getenv("QWEN_LLM_API_KEY", "").strip()
    or os.getenv("OPENAI_LLM_API_KEY", "").strip()
    or os.getenv("OPENAI_API_KEY", "").strip()
)

# OPENAI provider for eligibility/context validation
OPENAI_BASE_URL = os.getenv("OPENAI_LLM_API_URL", "").strip()
OPENAI_MODEL = os.getenv("OPENAI_LLM_MODEL_NAME", "openai/gpt-oss-20b").strip()
OPENAI_API_KEY = (
    os.getenv("OPENAI_LLM_API_KEY", "").strip()
    or os.getenv("OPENAI_API_KEY", "").strip()
)

WINDOW_SIZE = 5
WINDOW_STRIDE = 3
QUESTIONS_PER_WINDOW = 30  # robust profile: 6 Bloom levels x 5 formats
MAX_CONTEXT_CHARS = 18000
MIN_CONTEXT_CHARS = 450
GENERATION_MAX_RETRIES = 3
VALIDATION_MAX_RETRIES = 2
DEDUP_THRESHOLD = 90
SEED = 42

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Books dir: {BOOKS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"QWEN model: {QWEN_MODEL} | URL set: {bool(QWEN_BASE_URL)}")
print(f"OPENAI model: {OPENAI_MODEL} | URL set: {bool(OPENAI_BASE_URL)}")

python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 18
python-dotenv could not parse statement starting at line 19
python-dotenv could not parse statement starting at line 20
python-dotenv could not parse statement starting at line 21
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 24
python-dotenv could not parse statement starting at line 25
python-dotenv could not parse statement starting at line 26
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 28
python-dotenv could not parse statement starting at line 29
python-dotenv could not parse statement starting at line 30
python-dotenv could not parse statement starting at line 31
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 34
python-dotenv could not parse statement 

Workspace root: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning
Books dir: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/Books
Output dir: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training
QWEN model: Qwen/Qwen3.5-27B | URL set: True
OPENAI model: openai/gpt-oss-20b | URL set: True


In [3]:
import fitz


NOISE_PATTERNS = [
    r"^\s*\d+\s*$",
    r"^\s*page\s+\d+\s*$",
    r"^\s*chapter\s+\d+\s*$",
    r"^\s*contents\s*$",
    r"^\s*copyright\b.*$",
    r"^\s*all rights reserved\b.*$",
    r"^\s*isbn\b.*$",
    r"^\s*www\..*$",
    r"^\s*https?://.*$",
    r"^\s*\.{4,}\s*\d+\s*$",
]


def normalize_text(text: str) -> str:
    text = text or ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")
    text = re.sub(r"-\n(?=[a-z])", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def should_drop_line(line: str) -> bool:
    clean = line.strip()
    if not clean:
        return True
    if len(clean) <= 2:
        return True
    for pat in NOISE_PATTERNS:
        if re.match(pat, clean, flags=re.IGNORECASE):
            return True
    if sum(ch.isdigit() for ch in clean) > 10 and len(clean) < 30:
        return True
    return False


def deep_clean_text(raw_text: str) -> str:
    lines = [ln.rstrip() for ln in raw_text.splitlines()]
    cleaned: List[str] = []
    for line in lines:
        if should_drop_line(line):
            continue
        line = re.sub(r"\s+", " ", line).strip()
        cleaned.append(line)

    merged: List[str] = []
    for line in cleaned:
        if merged and len(line) < 80 and line[0].islower() and not merged[-1].endswith(('.', ':', ';', '?', '!')):
            merged[-1] = f"{merged[-1]} {line}"
        else:
            merged.append(line)

    text = "\n".join(merged)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pages_text(pdf_path: Path, page_start: int, page_end: int) -> str:
    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)
        s = max(1, page_start)
        e = min(page_end, total_pages)
        parts: List[str] = []

        for page_num in range(s, e + 1):
            page = doc[page_num - 1]
            h = page.rect.height
            blocks = page.get_text("blocks")
            body = [
                b[4]
                for b in blocks
                if b[6] == 0 and b[4].strip() and b[1] > h * 0.06 and b[3] < h * 0.94
            ]
            page_text = "\n".join(body)
            cleaned = deep_clean_text(page_text)
            if cleaned:
                parts.append(f"[Page {page_num}]\n{cleaned}")

    return "\n\n".join(parts).strip()


def get_pdf_page_count(pdf_path: Path) -> int:
    with fitz.open(pdf_path) as doc:
        return len(doc)

In [4]:
def build_qwen_client() -> OpenAI:
    if not QWEN_BASE_URL:
        raise ValueError("Set QWEN_LLM_API_URL and QWEN_LLM_API_KEY (or OPENAI_LLM_API_KEY) in environment/.env")
    return OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY)


def build_openai_validation_client() -> OpenAI:
    if not OPENAI_BASE_URL:
        raise ValueError("Set OPENAI_LLM_API_URL and OPENAI_LLM_API_KEY in environment/.env")
    return OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)


DIFFICULTY_SNIPPETS: Dict[str, str] = {
    "Remember": (
        "Target faithful retrieval of explicitly stated facts, definitions, symbols, constraints, or step names from the given context. "
        "Use action intent such as identify, list, recall, or state. "
        "Do not require inference beyond direct textual evidence. "
        "Answers should be short, exact, and traceable to a specific statement in the passage."
    ),
    "Understand": (
        "Target comprehension of meaning, relationships, and interpretation. "
        "Use intents like explain, summarize, classify, compare at a basic level, or interpret in plain language. "
        "Questions should verify that the learner can restate the concept accurately without adding external assumptions. "
        "Answers should include brief conceptual linkage, not mere copying."
    ),
    "Apply": (
        "Target transfer of a concept, rule, or procedure to a concrete mini-scenario consistent with the context. "
        "Use intents such as apply, compute, determine, execute, or choose based on stated rules. "
        "Require 1-2 explicit reasoning steps grounded in given conditions. "
        "Answers should demonstrate how the rule is used, not only the final outcome."
    ),
    "Analyze": (
        "Target decomposition and structural reasoning: part-whole relationships, assumptions, causal pathways, and distinctions. "
        "Use intents like differentiate, infer structure, diagnose, or compare mechanisms with evidence. "
        "Questions should require multi-part reasoning and discrimination between similar options. "
        "Answers should cite the decisive evidence from context."
    ),
    "Evaluate": (
        "Target judgment using explicit criteria present in the context, such as correctness, efficiency, robustness, or trade-offs. "
        "Use intents such as justify, critique, defend, prioritize, or assess alternatives. "
        "Questions must force a choice under constraints and require rationale for rejecting competing choices. "
        "Answers should present criterion-based justification, not opinion-only claims."
    ),
    "Create": (
        "Target synthesis of ideas into a new but context-consistent artifact: design choice, algorithm sketch, improved formulation, or structured plan. "
        "Use intents such as design, construct, propose, formulate, or compose. "
        "Questions should combine multiple concepts from the passage while preserving constraints and factual consistency. "
        "Answers should present a coherent constructed solution with concise rationale."
    ),
}

FORMAT_SNIPPETS: Dict[str, str] = {
    "MCQ": (
        "Include exactly four options (A-D), one unambiguously correct answer, "
        "and plausible distractors derived from common misconceptions in the same context."
    ),
    "Fill_blank": (
        "Use a single meaningful blank whose completion depends on conceptual understanding, "
        "not rote memorization of a random token."
    ),
    "Assertion": (
        "Use assertion-reason style with a clear verdict in the answer. "
        "The rationale must explicitly explain whether the reason supports the assertion."
    ),
    "Analytical": (
        "Ask for structured analysis or comparison (e.g., mechanism vs outcome, choice vs trade-off). "
        "Answer should be concise but logically sequenced."
    ),
    "Conceptual": (
        "Ask a self-contained explanatory question that tests understanding of principles, "
        "causality, or interpretation from the provided context."
    ),
}

SUBFORMAT_FOR_MAIN = {
    "Objective": ["MCQ", "Fill_blank", "Assertion", "Analytical"],
    "Subjective": ["Conceptual"],
}

DIFFICULTY_ORDER: List[str] = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
SUBFORMAT_ORDER: List[str] = ["MCQ", "Fill_blank", "Assertion", "Analytical", "Conceptual"]
SUBFORMAT_TO_MAIN_FORMAT: Dict[str, str] = {
    "MCQ": "Objective",
    "Fill_blank": "Objective",
    "Assertion": "Objective",
    "Analytical": "Objective",
    "Conceptual": "Subjective",
}

ALL_QUESTION_COMBINATIONS: List[Tuple[str, str, str]] = [
    (difficulty, SUBFORMAT_TO_MAIN_FORMAT[sub_format], sub_format)
    for sub_format in SUBFORMAT_ORDER
    for difficulty in DIFFICULTY_ORDER
]
DEFAULT_QUESTIONS_PER_VALID_WINDOW = len(ALL_QUESTION_COMBINATIONS)

# In-memory trace for all LLM calls in a run.
LLM_TRACE_LOGS: List[Dict[str, Any]] = []
QWEN_RAW_RESPONSE_LOG = os.getenv("QWEN_RAW_RESPONSE_LOG", "1").strip().lower() in {"1", "true", "yes", "on"}


def _preview(text: str, n: int = 180) -> str:
    text = (text or "").replace("\n", " ").strip()
    return text[:n] + ("..." if len(text) > n else "")


def _append_llm_trace(entry: Dict[str, Any]) -> None:
    LLM_TRACE_LOGS.append(entry)
    payload_txt = " | ".join(f"{k}={v}" for k, v in entry.items())
    print(f"[llm::summary] {payload_txt}")


def _log_qwen_raw_response(
    response_type: str,
    window_tag: str,
    attempt: int,
    variant_idx: int,
    finish_reason: Any,
    raw: str,
    reasoning: str,
) -> None:
    if not QWEN_RAW_RESPONSE_LOG:
        return
    print("[qwen::raw] ----- begin -----")
    print(f"[qwen::raw] type={response_type} window={window_tag or 'na'} attempt={attempt} variant={variant_idx} finish_reason={finish_reason}")
    print(f"[qwen::raw] content_chars={len(raw or '')} reasoning_chars={len(reasoning or '')}")
    print("[qwen::raw] content:")
    print(raw if raw else "<EMPTY>")
    if reasoning:
        print("[qwen::raw] reasoning_content:")
        print(reasoning)
    print("[qwen::raw] ----- end -----")


def _strip_qwen_think_blocks(text: str) -> str:
    # Qwen reasoning models may return <think>...</think> in assistant content.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL | re.IGNORECASE).strip()


def _extract_json_candidate(raw: str) -> str:
    cleaned = re.sub(r"```(?:json)?", "", (raw or "")).replace("```", "").strip()
    if not cleaned:
        return ""

    cleaned = _strip_qwen_think_blocks(cleaned)

    # Prefer object blocks first.
    m_obj = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if m_obj:
        return m_obj.group(0).strip()

    # Some models may emit a top-level list.
    m_list = re.search(r"\[.*\]", cleaned, flags=re.DOTALL)
    if m_list:
        return m_list.group(0).strip()

    return cleaned


def _extract_balanced_json_span(text: str) -> str:
    if not text:
        return ""
    start = -1
    stack: List[str] = []
    in_str = False
    esc = False
    for i, ch in enumerate(text):
        if start == -1 and ch in "[{":
            start = i
            stack = [ch]
            in_str = False
            esc = False
            continue
        if start == -1:
            continue
        if in_str:
            if esc:
                esc = False
                continue
            if ch == "\\":
                esc = True
                continue
            if ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
            continue
        if ch in "[{":
            stack.append(ch)
            continue
        if ch in "]}":
            if not stack:
                continue
            opener = stack[-1]
            if (opener == "[" and ch == "]") or (opener == "{" and ch == "}"):
                stack.pop()
                if not stack and start != -1:
                    return text[start:i+1].strip()
    return ""


def _strip_trailing_commas(text: str) -> str:
    prev = None
    cur = text
    while prev != cur:
        prev = cur
        cur = re.sub(r",\s*([}\]])", r"\1", cur)
    return cur


def _close_unbalanced_json(text: str) -> str:
    if not text:
        return text
    stack: List[str] = []
    in_str = False
    esc = False
    for ch in text:
        if in_str:
            if esc:
                esc = False
                continue
            if ch == "\\":
                esc = True
                continue
            if ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
            continue
        if ch in "[{":
            stack.append(ch)
        elif ch in "]}" and stack:
            opener = stack[-1]
            if (opener == "[" and ch == "]") or (opener == "{" and ch == "}"):
                stack.pop()
    closers = []
    for opener in reversed(stack):
        closers.append("]" if opener == "[" else "}")
    return text + "".join(closers)


def _extract_items_object_fallback(text: str) -> Dict[str, Any]:
    # Last-resort parser: pull per-item JSON objects from malformed arrays.
    items: List[Dict[str, Any]] = []
    for m in re.finditer(r"\{[^{}]*\}", text, flags=re.DOTALL):
        chunk = m.group(0).strip()
        if '"question"' not in chunk or '"answer"' not in chunk:
            continue
        chunk = _strip_trailing_commas(_close_unbalanced_json(chunk))
        try:
            obj = json.loads(chunk)
            if isinstance(obj, dict):
                items.append(obj)
        except Exception:
            continue
    return {"items": items}


def _parse_json_payload(candidate: str, response_type: str) -> Dict[str, Any]:
    if not candidate:
        raise ValueError("empty_response")

    attempts: List[str] = []
    attempts.append(candidate)
    span = _extract_balanced_json_span(candidate)
    if span:
        attempts.append(span)
    attempts.append(_strip_trailing_commas(candidate))
    attempts.append(_strip_trailing_commas(_close_unbalanced_json(candidate)))

    seen: set = set()
    for payload in attempts:
        payload = (payload or "").strip()
        if not payload or payload in seen:
            continue
        seen.add(payload)
        try:
            parsed = json.loads(payload)
        except Exception:
            continue

        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list) and response_type == "generation":
            return {"items": parsed}

    if response_type == "generation":
        fallback = _extract_items_object_fallback(candidate)
        if fallback.get("items"):
            return fallback

    raise ValueError("unparseable_json_payload")


def get_question_combinations(question_count: int) -> List[Tuple[str, str, str]]:
    base = list(ALL_QUESTION_COMBINATIONS)
    if question_count <= 0:
        return []
    if question_count <= len(base):
        return base[:question_count]

    expanded: List[Tuple[str, str, str]] = []
    while len(expanded) < question_count:
        expanded.extend(base)
    return expanded[:question_count]


def _combo_key(difficulty: str, main_format: str, sub_format: str) -> str:
    return f"{difficulty}|{main_format}|{sub_format}"


BLOOMS_ALIAS_MAP: Dict[str, str] = {
    "remember": "Remember",
    "knowledge": "Remember",
    "understand": "Understand",
    "understanding": "Understand",
    "comprehend": "Understand",
    "apply": "Apply",
    "application": "Apply",
    "analyze": "Analyze",
    "analysis": "Analyze",
    "analyse": "Analyze",
    "evaluate": "Evaluate",
    "evaluation": "Evaluate",
    "create": "Create",
    "creation": "Create",
    "synthesize": "Create",
    "synthesis": "Create",
}


def normalize_blooms_level(value: str) -> str:
    v = normalize_text(value).lower()
    return BLOOMS_ALIAS_MAP.get(v, "Understand")

GENERATION_ENABLE_THINKING = os.getenv("QWEN_ENABLE_THINKING", "0").strip().lower() in {"1", "true", "yes", "on"}

def _message_text(msg: Any) -> str:
    content = getattr(msg, "content", None)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts: List[str] = []
        for p in content:
            if isinstance(p, dict):
                t = p.get("text") or p.get("content") or ""
                if t:
                    parts.append(str(t))
        return "\n".join(parts).strip()
    return ""


def chat_json(
    client: OpenAI,
    model: str,
    system_prompt: str,
    user_prompt: str,
    max_tokens: Optional[int] = None,
    temperature: float = 0.2,
    response_type: str = "generic",
    window_tag: str = "",
    max_retries: int = 2,
) -> Dict[str, Any]:
    last_error: Optional[Exception] = None

    for attempt in range(max_retries + 1):
        strict_suffix = ""
        if attempt > 0:
            strict_suffix = (
                "\n\nIMPORTANT: Return ONLY valid JSON with no prose, no markdown, no code fences."
            )

        # First try with configured mode.
        variants: List[Dict[str, Any]] = []
        base_req: Dict[str, Any] = {
            "model": model,
            "temperature": 0.0 if attempt > 0 else temperature,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt + strict_suffix},
            ],
        }
        if max_tokens is not None:
            base_req["max_tokens"] = max_tokens if attempt == 0 else max(1800, min(max_tokens * 2, 4096))
        if response_type == "generation" and GENERATION_ENABLE_THINKING:
            base_req["extra_body"] = {"chat_template_kwargs": {"enable_thinking": True}}
        variants.append(base_req)

        # Fallback for generation: hard-disable thinking and give larger completion budget.
        if response_type == "generation":
            gen_fallback: Dict[str, Any] = {
                "model": model,
                "temperature": 0.0,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt + strict_suffix},
                ],
            }
            if max_tokens is not None:
                gen_fallback["max_tokens"] = max(2400, min(max_tokens * 2, 4096))
            variants.append(gen_fallback)

        for variant_idx, request_kwargs in enumerate(variants, start=1):
            try:
                resp = client.chat.completions.create(**request_kwargs)
            except Exception as exc:
                last_error = exc
                _append_llm_trace(
                    {
                        "type": response_type,
                        "window": window_tag or "na",
                        "model": model,
                        "status": "request_error",
                        "attempt": attempt + 1,
                        "variant": variant_idx,
                        "error": str(exc),
                    }
                )
                continue

            choice = resp.choices[0]
            msg = choice.message
            finish_reason = getattr(choice, "finish_reason", None)
            raw = _message_text(msg)
            reasoning_content = str(getattr(msg, "reasoning_content", None) or "")
            if model == QWEN_MODEL:
                _log_qwen_raw_response(
                    response_type=response_type,
                    window_tag=window_tag,
                    attempt=attempt + 1,
                    variant_idx=variant_idx,
                    finish_reason=finish_reason,
                    raw=raw,
                    reasoning=reasoning_content,
                )
            candidate = _extract_json_candidate(raw)

            if not candidate:
                last_error = ValueError("empty_response")
                _append_llm_trace(
                    {
                        "type": response_type,
                        "window": window_tag or "na",
                        "model": model,
                        "status": "empty_content",
                        "attempt": attempt + 1,
                        "variant": variant_idx,
                        "finish_reason": finish_reason,
                        "raw_chars": len(raw),
                        "reasoning_chars": len(reasoning_content),
                    }
                )
                continue

            try:
                data = _parse_json_payload(candidate, response_type=response_type)
            except Exception as exc:
                last_error = exc
                _append_llm_trace(
                    {
                        "type": response_type,
                        "window": window_tag or "na",
                        "model": model,
                        "status": "parse_error",
                        "attempt": attempt + 1,
                        "variant": variant_idx,
                        "finish_reason": finish_reason,
                        "raw_chars": len(raw),
                        "reasoning_chars": len(reasoning_content),
                        "raw_preview": _preview(raw),
                        "error": str(exc),
                    }
                )
                continue

            _append_llm_trace(
                {
                    "type": response_type,
                    "window": window_tag or "na",
                    "model": model,
                    "status": "ok",
                    "attempt": attempt + 1,
                    "variant": variant_idx,
                    "finish_reason": finish_reason,
                    "raw_chars": len(raw),
                    "reasoning_chars": len(reasoning_content),
                }
            )
            return data

    raise ValueError(f"json_parse_failed_after_retries::{last_error}")


def ask_eligibility(
    validation_client: OpenAI,
    context_text: str,
    window_tag: str = "",
    max_questions_per_page: int = 1,
    window_pages: int = WINDOW_SIZE,
    validation_retries: int = VALIDATION_MAX_RETRIES,
) -> Dict[str, Any]:
    system_prompt = (
        "You are a strict textbook-content gatekeeper for QA dataset creation. "
        "Your job is to reject noisy/non-teachable pages and allow only concept-rich instructional content. "
        "Return strict JSON only."
    )
    max_questions_per_page = max(1, int(max_questions_per_page))
    window_pages = max(1, int(window_pages))
    max_questions_for_window = max_questions_per_page * window_pages

    user_prompt = f"""
Context (trimmed):
{context_text[:MAX_CONTEXT_CHARS]}

Return JSON with keys:
- eligible: true/false
- reason: short snake_case reason
- recommended_questions: integer between 0 and {max_questions_for_window}

Primary objective:
- We run this validation mainly to remove noisy book sections (author index, acknowledgements, references, glossary, publication metadata, copyright pages, TOC pages, and similar non-teaching content).

Eligibility rules:
- Mark eligible=false when the context is primarily structural/noisy matter: author index entries, acknowledgements, bibliography/references, table of contents, preface-only notes, appendices that are lists/tables only, page headers/footers, or OCR garbage.
- Mark eligible=true only if the passage contains enough teachable material: definitions, explanations, procedures, comparisons, worked reasoning, or conceptual discussion that can support self-contained QA generation.
- If mixed content appears, prefer eligible=false unless at least ~60% of the text is instructional and concept-bearing.
- Keep reason concise and specific, e.g., author_index_page, acknowledgements_page, toc_page, bibliography_page, noisy_ocr, concept_rich_text.
- For eligible windows, use this ceiling rule:
  max_questions_per_page = {max_questions_per_page}
  window_pages = {window_pages}
  max_questions_for_window = {max_questions_for_window}
- Keep recommended_questions <= max_questions_for_window.
"""
    try:
        result = chat_json(
            client=validation_client,
            model=OPENAI_MODEL,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            max_tokens=280,
            temperature=0.0,
            response_type="validation",
            window_tag=window_tag,
            max_retries=max(1, int(validation_retries)),
        )
    except Exception as exc:
        _append_llm_trace(
            {
                "type": "validation",
                "window": window_tag or "na",
                "model": OPENAI_MODEL,
                "status": "fallback",
                "reason": str(exc),
            }
        )
        return {
            "eligible": True,
            "reason": f"eligibility_fallback::{exc}",
            "recommended_questions": max_questions_for_window,
        }

    return {
        "eligible": bool(result.get("eligible", True)),
        "reason": str(result.get("reason", "unknown")),
        "recommended_questions": int(result.get("recommended_questions", max_questions_for_window) or max_questions_for_window),
    }


def build_dynamic_prompt(context_text: str, combination_plan: List[Tuple[str, str, str]]) -> str:
    question_count = len(combination_plan)
    snippet_lines = []
    required_combo_lines = []

    for idx, (diff, main_fmt, sub_fmt) in enumerate(combination_plan, start=1):
        combo_id = f"combo_{idx:02d}_{diff.lower()}_{sub_fmt.lower()}"
        snippet_lines.append(
            f"- {combo_id}: {diff} | {main_fmt}/{sub_fmt}: {DIFFICULTY_SNIPPETS[diff]} {FORMAT_SNIPPETS[sub_fmt]}"
        )
        required_combo_lines.append(
            f"- {combo_id}: difficulty={diff}, main_format={main_fmt}, sub_format={sub_fmt}"
        )

    return f"""
Generate exactly {question_count} high-quality QA items from the context.

Context:
{context_text[:MAX_CONTEXT_CHARS]}

Hard constraints:
1) The examnee will NOT be given the source context; every question must be fully self-contained and understandable on its own.
2) Do not reference location markers like 'line 7', 'above paragraph', 'this page', 'in the figure', or 'section 1.2'.
3) Do not use arbitrary numeric references unless the context contains executable code/pseudocode and the number is semantically necessary.
4) Use only facts present in context. No external facts.
5) Avoid duplicates and near-duplicates.
6) Keep answers concise but complete.
7) Include a short reasoning field for each item that justifies why the answer is correct from context.
8) Keep reasoning concise (1-3 sentences), evidence-grounded, and free of chain-of-thought style hidden deliberations.
9) Produce exactly one item for each required combination below. Do not skip any combination.
10) Output compact valid JSON only. Do not add markdown, comments, trailing commas, or extra keys.
11) Each item must include all required keys and valid enum values.
12) Keep question and answer concise to avoid truncation; avoid long paragraphs.

Required combinations (exactly one item each):
{chr(10).join(required_combo_lines)}

Bloom taxonomy/format guidance:
{chr(10).join(snippet_lines)}

Field rules:
- combo_id: must exactly match one required combo_id above.
- difficulty: one of Remember, Understand, Apply, Analyze, Evaluate, Create.
- main_format: one of Objective, Subjective.
- sub_format: one of MCQ, Fill_blank, Assertion, Analytical, Conceptual.
- MCQ must provide exactly 4 options using either (A) embedded labels in question text OR (B) dedicated fields option_a, option_b, option_c, option_d.
- If dedicated option fields are used, question should be the stem only (no inline options).
- MCQ answer must start with correct option label (A/B/C/D) and then a brief rationale.
- Fill_blank question should contain exactly one blank token: ____.
- reasoning must be 1-2 short evidence-grounded sentences.
- Align each item with the cognitive process demanded by its Bloom level and use verbs naturally aligned to that level.

Return strict JSON with shape:
{{
  "items": [
    {{
      "combo_id": "combo_01_remember_mcq",
      "question": "...",
      "answer": "...",
      "reasoning": "brief grounded rationale",
      "difficulty": "Remember|Understand|Apply|Analyze|Evaluate|Create",
      "main_topic": "DSA",
      "sub_topic": "...",
      "main_format": "Objective|Subjective",
      "sub_format": "MCQ|Fill_blank|Assertion|Analytical|Conceptual",
      "option_a": "optional_for_mcq",
      "option_b": "optional_for_mcq",
      "option_c": "optional_for_mcq",
      "option_d": "optional_for_mcq",
      "correct_option": "A|B|C|D (optional_for_mcq)"
    }}
  ]
}}
"""


def _chunked(seq: List[Tuple[str, str, str]], size: int) -> List[List[Tuple[str, str, str]]]:
    return [seq[i:i + size] for i in range(0, len(seq), size)]


def _has_mcq_options(question_text: str) -> bool:
    q = normalize_text(question_text)
    return bool(
        re.search(r"(?:^|\s)A\)", q)
        and re.search(r"(?:^|\s)B\)", q)
        and re.search(r"(?:^|\s)C\)", q)
        and re.search(r"(?:^|\s)D\)", q)
    )


def _is_valid_item_for_format(sub_format: str, question: str, answer: str, item: Dict[str, Any]) -> bool:
    if sub_format == "MCQ":
        has_inline = _has_mcq_options(question)
        has_structured = all(normalize_text(str(item.get(k, ""))) for k in ["option_a", "option_b", "option_c", "option_d"])
        has_correct = bool(re.match(r"^\s*[ABCD]\)", normalize_text(answer), flags=re.IGNORECASE) or normalize_text(str(item.get("correct_option", ""))) in {"A", "B", "C", "D"})
        return (has_inline or has_structured) and has_correct
    if sub_format == "Fill_blank":
        return question.count("____") == 1
    if sub_format == "Assertion":
        q = normalize_text(question)
        return bool(re.search(r"Assertion\s*[:\-]", q, flags=re.IGNORECASE) and re.search(r"Reason\s*[:\-]", q, flags=re.IGNORECASE))
    return True


def generate_window_questions(
    generation_client: OpenAI,
    context_text: str,
    question_count: int,
    window_tag: str = "",
    combination_plan: Optional[List[Tuple[str, str, str]]] = None,
    generation_retries: int = GENERATION_MAX_RETRIES,
) -> List[Dict[str, Any]]:
    system_prompt = (
        "You are an expert computer science educator creating rigorous study QA data. "
        "Use internal reasoning, but return only strict JSON in the final content."
    )

    full_plan = combination_plan or get_question_combinations(question_count)
    if not full_plan:
        return []

    # Keep generation prompts lighter than full window context to reduce backend truncation risk.
    trimmed_context = (context_text or "")[:7000]

    # Combination plan is executed in batches of 5 to keep prompts stable.
    batch_size = 5 if len(full_plan) > 5 else len(full_plan)
    plan_batches = _chunked(full_plan, batch_size)

    all_rows: List[Dict[str, Any]] = []
    missing_total: List[str] = []

    for b_idx, batch_plan in enumerate(plan_batches, start=1):
        batch_tag = f"{window_tag}::batch{b_idx}/{len(plan_batches)}"
        user_prompt = build_dynamic_prompt(context_text=trimmed_context, combination_plan=batch_plan)
        expected_keys = {_combo_key(diff, main_fmt, sub_fmt) for diff, main_fmt, sub_fmt in batch_plan}

        retries = max(1, int(generation_retries))
        for batch_attempt in range(1, retries + 1):
            data = chat_json(
                client=generation_client,
                model=QWEN_MODEL,
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                temperature=0.1,
                response_type="generation",
                window_tag=f"{batch_tag}::retry{batch_attempt}",
                max_retries=max(1, int(generation_retries)),
            )

            items = data.get("items", [])
            if not isinstance(items, list):
                _append_llm_trace(
                    {
                        "type": "generation",
                        "window": batch_tag,
                        "model": QWEN_MODEL,
                        "status": "invalid_items_payload",
                        "batch_attempt": batch_attempt,
                    }
                )
                items = []

            normalized: List[Dict[str, Any]] = []
            for item in items:
                if not isinstance(item, dict):
                    continue

                q = normalize_text(str(item.get("question", "")))
                a = normalize_text(str(item.get("answer", "")))
                r = normalize_text(str(item.get("reasoning", "")))
                if not q or not a:
                    continue

                difficulty = normalize_blooms_level(str(item.get("difficulty", "Understand")))

                sub_format = str(item.get("sub_format", "Conceptual"))
                if sub_format not in SUBFORMAT_TO_MAIN_FORMAT:
                    sub_format = "Conceptual"

                if not _is_valid_item_for_format(sub_format, q, a, item):
                    _append_llm_trace(
                        {
                            "type": "generation",
                            "window": batch_tag,
                            "model": QWEN_MODEL,
                            "status": "invalid_format_item",
                            "batch_attempt": batch_attempt,
                            "sub_format": sub_format,
                            "question_preview": _preview(q),
                        }
                    )
                    continue

                mapped_main_format = SUBFORMAT_TO_MAIN_FORMAT[sub_format]
                combo_key = _combo_key(difficulty, mapped_main_format, sub_format)

                normalized.append(
                    {
                        "combo_id": str(item.get("combo_id", "")) or "",
                        "combo_key": combo_key,
                        "question": q,
                        "answer": a,
                        "reasoning": r,
                        "difficulty": difficulty,
                        "main_topic": str(item.get("main_topic", "DSA")) or "DSA",
                        "sub_topic": str(item.get("sub_topic", "General")) or "General",
                        "main_format": mapped_main_format,
                        "sub_format": sub_format,
                        "option_a": normalize_text(str(item.get("option_a", ""))),
                        "option_b": normalize_text(str(item.get("option_b", ""))),
                        "option_c": normalize_text(str(item.get("option_c", ""))),
                        "option_d": normalize_text(str(item.get("option_d", ""))),
                        "correct_option": normalize_text(str(item.get("correct_option", ""))).upper(),
                    }
                )

            selected_by_combo: Dict[str, Dict[str, Any]] = {}
            for row in normalized:
                key = row["combo_key"]
                if key not in expected_keys:
                    continue
                if key in selected_by_combo:
                    continue
                selected_by_combo[key] = row

            missing_batch: List[str] = []
            tentative_rows: List[Dict[str, Any]] = []
            for diff, main_fmt, sub_fmt in batch_plan:
                key = _combo_key(diff, main_fmt, sub_fmt)
                row = selected_by_combo.get(key)
                if row:
                    row = dict(row)
                    row.pop("combo_key", None)
                    tentative_rows.append(row)
                else:
                    missing_batch.append(key)

            if not missing_batch:
                all_rows.extend(tentative_rows)
                break

            _append_llm_trace(
                {
                    "type": "generation",
                    "window": batch_tag,
                    "model": QWEN_MODEL,
                    "status": "batch_retry_validation_failed",
                    "batch_attempt": batch_attempt,
                    "missing_combos": len(missing_batch),
                }
            )

            if batch_attempt == retries:
                all_rows.extend(tentative_rows)
                missing_total.extend(missing_batch)

    _append_llm_trace(
        {
            "type": "generation",
            "window": window_tag or "na",
            "model": QWEN_MODEL,
            "status": "batched_normalized",
            "requested": len(full_plan),
            "normalized_items": len(all_rows),
            "missing_combos": len(missing_total),
            "batches": len(plan_batches),
        }
    )

    if missing_total:
        print(f"[window] {window_tag} missing_combinations={len(missing_total)} details={missing_total}")

    return all_rows

In [5]:
# Deep smoke matrix: compare multiple request shapes and inspect raw choice payload.
# Goal: detect whether empty content is tied to thinking mode or server behavior.

from pprint import pprint


def _msg_to_text(msg: Any) -> str:
    content = getattr(msg, "content", None)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        # Some backends return OpenAI content parts.
        parts: List[str] = []
        for p in content:
            if isinstance(p, dict):
                t = p.get("text") or p.get("content") or ""
                if t:
                    parts.append(str(t))
            elif hasattr(p, "text"):
                t = getattr(p, "text", "")
                if t:
                    parts.append(str(t))
        return "\n".join(parts).strip()
    return ""


qwen_client = build_qwen_client()
base_messages = [
    {"role": "system", "content": "Return strict JSON only."},
    {"role": "user", "content": "Return exactly: {\"ok\": true, \"source\": \"qwen_smoke\"}"},
]

cases = [
    {"name": "think_on_temp00_unbounded", "temperature": 0.0, "enable_thinking": True},
    {"name": "think_off_temp00_unbounded", "temperature": 0.0, "enable_thinking": False},
    {"name": "think_off_temp02_unbounded", "temperature": 0.2, "enable_thinking": False},
]

results: List[Dict[str, Any]] = []

for cfg in cases:
    req = {
        "model": QWEN_MODEL,
        "messages": base_messages,
        "temperature": cfg["temperature"],
    }
    if cfg["enable_thinking"]:
        req["extra_body"] = {"chat_template_kwargs": {"enable_thinking": True}}

    try:
        resp = qwen_client.chat.completions.create(**req)
        choice = resp.choices[0]
        msg = choice.message
        content_text = _msg_to_text(msg)
        reasoning_text = str(getattr(msg, "reasoning_content", None) or "")
        _log_qwen_raw_response(
            response_type="smoke",
            window_tag=cfg["name"],
            attempt=1,
            variant_idx=1,
            finish_reason=getattr(choice, "finish_reason", None),
            raw=content_text,
            reasoning=reasoning_text,
        )
        usage = getattr(resp, "usage", None)

        row = {
            "case": cfg["name"],
            "finish_reason": getattr(choice, "finish_reason", None),
            "content_len": len(content_text),
            "reasoning_len": len(reasoning_text),
            "content_preview": content_text[:120],
            "reasoning_preview": reasoning_text[:120],
            "usage": usage.model_dump() if hasattr(usage, "model_dump") else str(usage),
        }
        results.append(row)

        print("\n===", cfg["name"], "===")
        print("finish_reason:", row["finish_reason"])
        print("content_len:", row["content_len"], "| reasoning_len:", row["reasoning_len"])
        print("content_preview:", repr(row["content_preview"]))
        print("reasoning_preview:", repr(row["reasoning_preview"]))

        # Print raw message for one failing case to inspect backend fields.
        if row["content_len"] == 0:
            print("raw_message_dump:")
            if hasattr(msg, "model_dump"):
                pprint(msg.model_dump())
            else:
                pprint(str(msg))

    except Exception as exc:
        results.append({"case": cfg["name"], "error": str(exc)})
        print("\n===", cfg["name"], "===")
        print("error:", exc)

print("\nSummary:")
for r in results:
    print(r)

[qwen::raw] ----- begin -----
[qwen::raw] type=smoke window=think_on_temp00_unbounded attempt=1 variant=1 finish_reason=stop
[qwen::raw] content_chars=794 reasoning_chars=0
[qwen::raw] content:
Thinking Process:

1.  **Analyze the Request:**
    *   Input: "Return exactly: {"ok": true, "source": "qwen_smoke"}"
    *   Constraint: "Return strict JSON only."
    *   Desired Output: A JSON object with keys "ok" (boolean true) and "source" (string "qwen_smoke").

2.  **Verify Constraints:**
    *   Must be valid JSON.
    *   Must not contain any markdown formatting (like ```json ... ```).
    *   Must not contain any extra text or explanations.
    *   Must match the specified structure exactly.

3.  **Construct Output:**
    *   `{"ok": true, "source": "qwen_smoke"}`

4.  **Final Check:**
    *   Is it valid JSON? Yes.
    *   Is it strict JSON only? Yes.
    *   Does it match the requested content? Yes.

5.  **Generate Output.**
</think>

{"ok": true, "source": "qwen_smoke"}
[qwen::raw]

In [6]:
from rapidfuzz import fuzz


def build_windows(total_pages: int, window_size: int = WINDOW_SIZE, stride: int = WINDOW_STRIDE) -> List[Tuple[int, int]]:
    if total_pages <= 0:
        return []
    windows: List[Tuple[int, int]] = []
    start = 1
    while start <= total_pages:
        end = min(total_pages, start + window_size - 1)
        windows.append((start, end))
        if end == total_pages:
            break
        start += stride
    return windows


def normalize_for_dedup(text: str) -> str:
    text = normalize_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def deduplicate_rows(rows: List[Dict[str, Any]], threshold: int = 92) -> List[Dict[str, Any]]:
    kept: List[Dict[str, Any]] = []
    seen: List[str] = []
    for row in rows:
        q_norm = normalize_for_dedup(row.get("question", ""))
        if not q_norm:
            continue

        if q_norm in seen:
            continue

        duplicate = False
        for prev in seen:
            if fuzz.token_set_ratio(q_norm, prev) >= threshold:
                duplicate = True
                break

        if duplicate:
            continue

        seen.append(q_norm)
        kept.append(row)
    return kept


def generate_for_book(
    generation_client: OpenAI,
    validation_client: OpenAI,
    pdf_path: Path,
    questions_per_window: int = QUESTIONS_PER_WINDOW,
    window_size: int = WINDOW_SIZE,
    stride: int = WINDOW_STRIDE,
    batch_number: int = 1,
    dedup_threshold: int = DEDUP_THRESHOLD,
    min_context_chars: int = MIN_CONTEXT_CHARS,
    generation_retries: int = GENERATION_MAX_RETRIES,
    validation_retries: int = VALIDATION_MAX_RETRIES,
) -> List[Dict[str, Any]]:
    total_pages = get_pdf_page_count(pdf_path)
    windows = build_windows(total_pages=total_pages, window_size=window_size, stride=stride)
    rows: List[Dict[str, Any]] = []
    target_questions_per_window = max(1, min(int(questions_per_window), len(ALL_QUESTION_COMBINATIONS)))

    print(f"[book] start={pdf_path.name} total_pages={total_pages} windows={len(windows)}")

    for w_idx, (p_start, p_end) in enumerate(tqdm(windows, desc=f"Windows::{pdf_path.name}"), start=1):
        window_tag = f"{pdf_path.name}:{p_start}-{p_end}"
        context_text = extract_pages_text(pdf_path=pdf_path, page_start=p_start, page_end=p_end)
        if not context_text or len(context_text) < max(200, int(min_context_chars)):
            print(f"[window] {window_tag} context_valid=False reason=insufficient_context chars={len(context_text or '')}")
            continue

        print(f"[window] {window_tag} context_valid=True chars={len(context_text)}")
        window_pages = max(1, p_end - p_start + 1)
        max_questions_per_page = max(1, (target_questions_per_window + max(1, window_size) - 1) // max(1, window_size))

        gate = ask_eligibility(
            validation_client,
            context_text,
            window_tag=window_tag,
            max_questions_per_page=max_questions_per_page,
            window_pages=window_pages,
            validation_retries=validation_retries,
        )
        if not gate["eligible"]:
            print(f"[window] {window_tag} eligible=False reason={gate.get('reason', 'unknown')}")
            continue

        combination_plan = list(get_question_combinations(target_questions_per_window))
        q_count = len(combination_plan)
        print(
            f"[window] {window_tag} eligible=True "
            f"recommended={gate.get('recommended_questions', 'na')} "
            f"requested_questions={q_count} "
            f"all_combinations={q_count == len(ALL_QUESTION_COMBINATIONS)}"
        )

        try:
            items = generate_window_questions(
                generation_client,
                context_text=context_text,
                question_count=q_count,
                window_tag=window_tag,
                combination_plan=combination_plan,
                generation_retries=generation_retries,
            )
        except Exception as exc:
            print(f"[warn] generation failed for {pdf_path.name} pages {p_start}-{p_end}: {exc}")
            continue

        print(f"[window] {window_tag} generated_items={len(items)}")

        for i, item in enumerate(items, start=1):
            rows.append(
                {
                    **item,
                    "book_name": pdf_path.name,
                    "page_number_range": f"{p_start} - {p_end}",
                    "page_start": p_start,
                    "page_end": p_end,
                    "window_index": w_idx,
                    "window_size": window_size,
                    "window_stride": stride,
                    "question_number": i,
                    "batch_number": batch_number,
                    "context_char_len": len(context_text),
                    "eligibility_reason": gate.get("reason", "unknown"),
                }
            )

    pre_dedup = len(rows)
    deduped = deduplicate_rows(rows, threshold=dedup_threshold)
    print(f"[book] complete={pdf_path.name} pre_dedup={pre_dedup} post_dedup={len(deduped)} removed={pre_dedup - len(deduped)}")

    return deduped


def _extract_mcq_fields(question: str, answer: str) -> Dict[str, Any]:
    q = normalize_text(question)
    # Parse A)/B)/C)/D) options into explicit fields.
    m = re.search(r"\bA\)\s*(.*?)\s*B\)\s*(.*?)\s*C\)\s*(.*?)\s*D\)\s*(.*)$", q, flags=re.IGNORECASE)
    if not m:
        return {
            "question_stem": q,
            "option_a": "",
            "option_b": "",
            "option_c": "",
            "option_d": "",
            "correct_option": "",
        }

    stem = q[:m.start()].strip(" :-")
    ans_match = re.match(r"^\s*([ABCD])\)", normalize_text(answer), flags=re.IGNORECASE)
    return {
        "question_stem": stem,
        "option_a": normalize_text(m.group(1)),
        "option_b": normalize_text(m.group(2)),
        "option_c": normalize_text(m.group(3)),
        "option_d": normalize_text(m.group(4)),
        "correct_option": ans_match.group(1).upper() if ans_match else "",
    }


def _extract_assertion_fields(question: str) -> Dict[str, Any]:
    q = normalize_text(question)
    m = re.search(r"Assertion\s*[:\-]\s*(.*?)\s*Reason\s*[:\-]\s*(.*)$", q, flags=re.IGNORECASE)
    if m:
        return {
            "assertion_statement": normalize_text(m.group(1)),
            "reason_statement": normalize_text(m.group(2)),
        }
    return {
        "assertion_statement": q,
        "reason_statement": "",
    }


def _enrich_row_for_jsonl(row: Dict[str, Any]) -> Dict[str, Any]:
    r = dict(row)
    sub_format = str(r.get("sub_format", "")).strip()
    question = str(r.get("question", ""))
    answer = str(r.get("answer", ""))

    if sub_format == "MCQ":
        parsed = _extract_mcq_fields(question, answer)
        for fld in ["question_stem", "option_a", "option_b", "option_c", "option_d", "correct_option"]:
            existing = normalize_text(str(r.get(fld, "")))
            if existing:
                parsed[fld] = existing
        if not parsed.get("correct_option"):
            m = re.match(r"^\s*([ABCD])\)", normalize_text(answer), flags=re.IGNORECASE)
            parsed["correct_option"] = m.group(1).upper() if m else ""
        r.update(parsed)
    elif sub_format == "Assertion":
        r.update(_extract_assertion_fields(question))
    elif sub_format == "Fill_blank":
        r["question_stem"] = question
        r["blank_count"] = question.count("____")
        r["expected_fill"] = normalize_text(answer)
    else:
        r["question_stem"] = question

    return r


def save_outputs(rows: List[Dict[str, Any]], stem: str) -> Dict[str, Path]:
    jsonl_path = OUTPUT_DIR / f"{stem}.jsonl"
    structured_rows = [_enrich_row_for_jsonl(r) for r in rows]

    with jsonl_path.open("w", encoding="utf-8") as f:
        for r in structured_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # Persist detailed LLM trace summaries for audit/debug.
    if LLM_TRACE_LOGS:
        trace_jsonl_path = OUTPUT_DIR / f"{stem}_llm_trace.jsonl"
        with trace_jsonl_path.open("w", encoding="utf-8") as f:
            for e in LLM_TRACE_LOGS:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
        print(f"[trace] wrote_llm_trace={trace_jsonl_path} rows={len(LLM_TRACE_LOGS)}")

    return {"jsonl": jsonl_path}

In [7]:
@dataclass
class RunConfig:
    questions_per_window: int = QUESTIONS_PER_WINDOW
    window_size: int = WINDOW_SIZE
    stride: int = WINDOW_STRIDE
    dedup_threshold: int = DEDUP_THRESHOLD
    min_context_chars: int = MIN_CONTEXT_CHARS
    generation_retries: int = GENERATION_MAX_RETRIES
    validation_retries: int = VALIDATION_MAX_RETRIES


RUN_CFG = RunConfig(
    questions_per_window=min(30, len(ALL_QUESTION_COMBINATIONS)),
    window_size=5,
    stride=2,
    dedup_threshold=90,
    min_context_chars=450,
    generation_retries=3,
    validation_retries=2,
)

print("Run configuration:")
print(RUN_CFG)

Run configuration:
RunConfig(questions_per_window=30, window_size=5, stride=2, dedup_threshold=90, min_context_chars=450, generation_retries=3, validation_retries=2)


## Section A: Single-Book Generation

Uses the shared API + cleaning + eligibility + sliding-window pipeline to generate QA rows for one selected PDF in the Books folder.

In [ ]:
# Section A: Generate QA for one selected book

generation_client = build_qwen_client()
validation_client = build_openai_validation_client()

pdf_candidates = sorted(BOOKS_DIR.glob("*.pdf"))
if not pdf_candidates:
    raise FileNotFoundError(f"No PDF files found in {BOOKS_DIR}")

TARGET_BOOK = pdf_candidates[0]  # change this to any specific file in BOOKS_DIR
print(f"Selected book: {TARGET_BOOK.name}")

single_rows = generate_for_book(
    generation_client=generation_client,
    validation_client=validation_client,
    pdf_path=TARGET_BOOK,
    questions_per_window=RUN_CFG.questions_per_window,
    window_size=RUN_CFG.window_size,
    stride=RUN_CFG.stride,
    batch_number=1,
    dedup_threshold=RUN_CFG.dedup_threshold,
    min_context_chars=RUN_CFG.min_context_chars,
    generation_retries=RUN_CFG.generation_retries,
    validation_retries=RUN_CFG.validation_retries,
)

single_paths = save_outputs(single_rows, stem=f"qa_{TARGET_BOOK.stem}_single_book")

print("Single-book generation complete.")
print(f"Rows: {len(single_rows)}")
for k, v in single_paths.items():
    print(f"- {k}: {v}")

Selected book: CAO - 1.pdf
[book] start=CAO - 1.pdf total_pages=1137 windows=567


Windows::CAO - 1.pdf:   0%|          | 0/567 [00:00<?, ?it/s]

[window] CAO - 1.pdf:1-5 context_valid=True chars=6384
[llm::summary] type=validation | window=CAO - 1.pdf:1-5 | model=openai/gpt-oss-20b | status=ok | attempt=1 | variant=1 | finish_reason=stop | raw_chars=92 | reasoning_chars=863
[window] CAO - 1.pdf:1-5 eligible=False reason=promotional_and_biographical_content
[window] CAO - 1.pdf:3-7 context_valid=True chars=7172
[llm::summary] type=validation | window=CAO - 1.pdf:3-7 | model=openai/gpt-oss-20b | status=ok | attempt=1 | variant=1 | finish_reason=stop | raw_chars=73 | reasoning_chars=460
[window] CAO - 1.pdf:3-7 eligible=False reason=front_matter_page
[window] CAO - 1.pdf:5-9 context_valid=True chars=6028
[llm::summary] type=validation | window=CAO - 1.pdf:5-9 | model=openai/gpt-oss-20b | status=ok | attempt=1 | variant=1 | finish_reason=stop | raw_chars=64 | reasoning_chars=428
[window] CAO - 1.pdf:5-9 eligible=False reason=toc_page
[window] CAO - 1.pdf:7-11 context_valid=True chars=5452
[llm::summary] type=validation | window=CAO

## Section B: All-Books Dataset Build

Reuses the exact same functions from Section A, iterates all PDFs in the Books folder, and exports one deduplicated combined dataset.

In [ ]:
# Section B: Generate QA for all books in BOOKS_DIR using the same pipeline

generation_client = build_qwen_client()
validation_client = build_openai_validation_client()
pdf_candidates = sorted(BOOKS_DIR.glob("*.pdf"))
if not pdf_candidates:
    raise FileNotFoundError(f"No PDF files found in {BOOKS_DIR}")

all_rows: List[Dict[str, Any]] = []
for batch_idx, pdf_path in enumerate(pdf_candidates, start=1):
    print(f"\nProcessing {batch_idx}/{len(pdf_candidates)}: {pdf_path.name}")
    book_rows = generate_for_book(
        generation_client=generation_client,
        validation_client=validation_client,
        pdf_path=pdf_path,
        questions_per_window=RUN_CFG.questions_per_window,
        window_size=RUN_CFG.window_size,
        stride=RUN_CFG.stride,
        batch_number=batch_idx,
        dedup_threshold=RUN_CFG.dedup_threshold,
        min_context_chars=RUN_CFG.min_context_chars,
        generation_retries=RUN_CFG.generation_retries,
        validation_retries=RUN_CFG.validation_retries,
    )
    print(f"  kept rows after dedup: {len(book_rows)}")
    all_rows.extend(book_rows)

# cross-book dedup to remove near-identical questions repeated across books/windows
all_rows = deduplicate_rows(all_rows, threshold=RUN_CFG.dedup_threshold)
all_paths = save_outputs(all_rows, stem="qa_all_books_sliding_window")

print("\nAll-books generation complete.")
print(f"Total rows: {len(all_rows)}")
for k, v in all_paths.items():
    print(f"- {k}: {v}")

if all_rows:
    df = pd.DataFrame(all_rows)
    print("\nDifficulty distribution:")
    print(df["difficulty"].value_counts().to_string())
    print("\nFormat distribution:")
    print((df["main_format"] + " / " + df["sub_format"]).value_counts().to_string())
    print("\nBooks coverage:")
    print(df["book_name"].value_counts().to_string())